# 01_build_panel
Build the person-level longitudinal panel from KHPS IND files (2019-2024).
Steps: load all waves, recode actionable features / covariates / targets, derive
BMI, age groups, income quintiles; assemble long-format panel keyed on PIDWON;
construct adjacent (t,t+1) and 2-year (t,t+2) transition frames with disease onset.
Outputs parquet files consumed by notebooks 02-05 and a cohort summary table.

In [1]:
%run 00_config.ipynb

PROJ_DIR: /home/claude/recourse_khp
1y pairs: [(2019, 2020), (2020, 2021), (2021, 2022), (2022, 2023), (2023, 2024)]
2y pairs: [(2019, 2021), (2020, 2022), (2021, 2023), (2022, 2024)]
registry loaded
helpers loaded
00_config ready


In [2]:
# --- Columns to pull from each IND wave (common set; wave-specific added later) ---
PULL = [KEY, HHKEY, "SEX","BIRTH_Y","EDU","H_INC_TOT","REGION1","DISA_YN",
        "HT","WT","D6","D7","P1","P2",
        "CD1_HTN","CD2_HTN","CD1_DM","CD2_DM","CD1_DYS","CD2_DYS",
        "ERGUN","INGUN","OUGUN","EROOP","INOOP","OUOOP_1","IN1YEAR",
        "I_PHI1_YN","I_FFS_YN","AH1","I_WSL"]
print("PULL base size:", len(PULL))

PULL base size: 31


In [3]:
# --- Columns present in EVERY wave stay in the common pull; wave-specific
# variables (e.g. D1 vs D1_2) are loaded per-wave when available. ---
def available_cols(wave):
    path=os.path.join(DATA_DIR, f"{wave}_ind.sas7bdat")
    _,meta=pyreadstat.read_sas7bdat(path, metadataonly=True)
    return set(meta.column_names)

WAVE_COLS={w:available_cols(w) for w in WAVE_SEQ}
common=set(PULL)
for w in WAVE_SEQ: common &= WAVE_COLS[w]
# wave-specific extras we still want if present in a given wave
WAVE_SPECIFIC=["D1","D1_2"]
print("cols in every wave:", len(common), "| wave-specific tracked:", WAVE_SPECIFIC)

cols in every wave: 27 | wave-specific tracked: ['D1', 'D1_2']


In [4]:
# --- Load and stack all waves; pull common cols + any wave-specific extras present ---
frames=[]
for w in WAVE_SEQ:
    cols=[c for c in PULL if c in WAVE_COLS[w]]
    for extra in WAVE_SPECIFIC:
        if extra in WAVE_COLS[w] and extra not in cols:
            cols.append(extra)
    df=load_ind(w, usecols=cols)
    frames.append(df)
raw=pd.concat(frames, ignore_index=True)   # missing cols -> NaN in waves lacking them
print("stacked shape:", raw.shape)
print(raw.groupby("year")[KEY].nunique())

stacked shape: (86569, 36)
year
2019    16587
2020    14844
2021    13799
2022    12909
2023    12169
2024    16261
Name: PIDWON, dtype: int64


In [5]:
# --- Recode into analysis variables ---
p=raw.copy()
# demographics
p["SEX"]  = p["SEX"].map({1:"M",2:"F"})
p["AGEG"] = age_group(p["age"])
p["INCQ"] = income_quintile(p["H_INC_TOT"])
p["EDU"]  = clean_missing(p["EDU"])
# actionable
p["BMI"]      = compute_bmi(p)
# Harmonize drinking frequency: D1 (2019-2022) and D1_2 (2023-2024) are the same
# question with different names AND different code sets. Collapse both to an ordinal
# 0=none .. higher=more frequent so the scale is comparable across waves.
def harmonize_alcfreq(row):
    v1=row.get("D1"); v2=row.get("D1_2")
    v = v1 if (pd.notna(v1) and v1 not in MISSING_CODES) else \
        (v2 if (pd.notna(v2) and v2 not in MISSING_CODES) else np.nan)
    if pd.isna(v): return np.nan
    # both scales: 1 = no drinking in ref period -> 0; increasing code -> more frequent
    return float(v)-1.0
p["ALC_FREQ"] = p.apply(harmonize_alcfreq, axis=1)
p["ALC_FREQ_SRC"] = np.where(p["D1"].notna() & ~p["D1"].isin(MISSING_CODES), "D1",
                     np.where(p.get("D1_2").notna() & ~p.get("D1_2").isin(MISSING_CODES),"D1_2","NA"))
p["ALC_AMT"]  = clean_missing(p["D6"])   if "D6" in p else np.nan
p["ALC_BINGE"]= clean_missing(p["D7"])   if "D7" in p else np.nan
p["PA_REG"]   = p["P1"].map({1:1,2:0})   if "P1" in p else np.nan   # 1=regular exercise
p["PA_WALK"]  = clean_missing(p["P2"])   if "P2" in p else np.nan
# targets: keep only those whose CD1/CD2 columns are present in the stacked frame
AVAIL_TARGETS={t:(c1,c2) for t,(c1,c2) in TARGETS.items() if c1 in p.columns and c2 in p.columns}
print("targets usable longitudinally:", list(AVAIL_TARGETS))
for t,(c1,c2) in AVAIL_TARGETS.items():
    p[f"{t}_dx"]   = yn_disease(p[c1])            # 1 has / 0 no at this wave
    p[f"{t}_dxtm"] = clean_missing(p[c2])         # diagnosis-timing code
# actuarial (fill utilisation NaN->0 handled downstream; keep raw here)
for v in ["ERGUN","INGUN","OUGUN","EROOP","INOOP","OUOOP_1"]:
    if v in p: p[v]=clean_missing(p[v])
for v in ["IN1YEAR","I_PHI1_YN","I_FFS_YN"]:
    if v in p: p[v]=p[v].map({1:1,2:0})
p["UNMET"]= p["AH1"].map({1:1,2:0}) if "AH1" in p else np.nan
print("recoded. adult (age>=19) share:", round((p.age>=19).mean(),3))

targets usable longitudinally: ['HTN', 'DM', 'DYS']
recoded. adult (age>=19) share: 0.788


In [6]:
# --- Restrict to adults with core actionable data, save long panel ---
panel = p[p.age>=19].copy()
base=[KEY,HHKEY,"wave","year","age","SEX","AGEG","EDU","INCQ","REGION1","DISA_YN",
      "H_INC_TOT","BMI","ALC_FREQ","ALC_AMT","ALC_BINGE","PA_REG","PA_WALK",
      "ERGUN","INGUN","OUGUN","EROOP","INOOP","OUOOP_1","IN1YEAR",
      "I_PHI1_YN","I_FFS_YN","UNMET","I_WSL"]
tgt_cols=[f"{t}_{sfx}" for t in AVAIL_TARGETS for sfx in ("dx","dxtm")]
core_cols=[c for c in base+tgt_cols if c in panel.columns]
panel=panel[core_cols]
panel.to_parquet(os.path.join(DATA_DIR,"panel_long.parquet"), index=False)
print("panel_long saved:", panel.shape)
panel.head(3)

panel_long saved: (68211, 35)


,PIDWON,HHID,wave,year,age,SEX,AGEG,EDU,INCQ,REGION1,...,I_PHI1_YN,I_FFS_YN,UNMET,I_WSL,HTN_dx,HTN_dxtm,DM_dx,DM_dxtm,DYS_dx,DYS_dxtm
0,11200101.0,112001011.0,a,2019,81.0,F,70+,1.0,Q1,26.0,...,0.0,NaN,NaN,NaN,1.0,5.0,0.0,NaN,NaN,NaN
1,11200201.0,112002011.0,a,2019,73.0,F,70+,1.0,Q1,26.0,...,0.0,NaN,NaN,NaN,0.0,NaN,0.0,NaN,NaN,NaN
3,11200301.0,112003011.0,a,2019,44.0,M,40-49,4.0,Q3,26.0,...,1.0,1.0,NaN,NaN,0.0,NaN,0.0,NaN,NaN,NaN


In [7]:
# --- Build transition frames (t -> t+1 and t -> t+2) with disease onset ---
def make_transitions(panel, pairs, tag):
    out=[]
    tgt=list(AVAIL_TARGETS)
    keep_t0=["BMI","ALC_FREQ","ALC_AMT","ALC_BINGE","PA_REG","PA_WALK",
             "age","SEX","AGEG","EDU","INCQ","REGION1","DISA_YN","H_INC_TOT"] \
             + (["I_WSL"] if "I_WSL" in panel.columns else []) \
             + [f"{t}_dx" for t in tgt]
    keep_t1=["BMI","ALC_FREQ","ALC_AMT","ALC_BINGE","PA_REG","PA_WALK",
             "ERGUN","INGUN","OUGUN","EROOP","INOOP","OUOOP_1","IN1YEAR",
             "I_PHI1_YN","I_FFS_YN","UNMET"] \
             + [f"{t}_dx" for t in tgt] + [f"{t}_dxtm" for t in tgt]
    keep_t0=[c for c in keep_t0 if c in panel.columns]
    keep_t1=[c for c in dict.fromkeys(keep_t1) if c in panel.columns]
    for w0,w1 in pairs:
        a=panel[panel.wave==w0].set_index(KEY)[keep_t0].add_suffix("_t0")
        b=panel[panel.wave==w1].set_index(KEY)[keep_t1].add_suffix("_t1")
        m=a.join(b, how="inner").reset_index()
        m["pair"]=f"{WAVES[w0]}_{WAVES[w1]}"; m["year0"]=WAVES[w0]; m["year1"]=WAVES[w1]
        out.append(m)
    tr=pd.concat(out, ignore_index=True)
    # onset per target: disease-free at t0 (dx==0) and has disease at t1 (dx==1)
    for t in AVAIL_TARGETS:
        atrisk=(tr[f"{t}_dx_t0"]==0)
        onset =atrisk & (tr[f"{t}_dx_t1"]==1)
        tr[f"{t}_atrisk"]=atrisk.astype(int)
        tr[f"{t}_onset"]=np.where(atrisk, onset.astype(int), np.nan)
    tr.to_parquet(os.path.join(DATA_DIR,f"transitions_{tag}.parquet"), index=False)
    return tr

tr1=make_transitions(panel, WAVE_PAIRS,    "1y")
tr2=make_transitions(panel, WAVE_PAIRS_2Y, "2y")
print("transitions_1y:", tr1.shape, "| transitions_2y:", tr2.shape)

transitions_1y: (49284, 50) | transitions_2y: (37809, 50)


In [8]:
# --- Cohort / onset summary table (primary target = HTN) ---
def onset_summary(tr, tag):
    rows=[]
    for pr,g in tr.groupby("pair"):
        ar=int(g[f"{PRIMARY_TARGET}_atrisk"].sum())
        on=int(np.nansum(g[f"{PRIMARY_TARGET}_onset"]))
        rows.append({"horizon":tag,"pair":pr,"linked_n":len(g),
                     "at_risk":ar,"onset":on,
                     "incidence_%":round(100*on/max(ar,1),2)})
    return pd.DataFrame(rows)
summ=pd.concat([onset_summary(tr1,"1y"), onset_summary(tr2,"2y")], ignore_index=True)
pooled=(summ.groupby("horizon")[["at_risk","onset"]].sum()
        .assign(incidence_pct=lambda d:(100*d.onset/d.at_risk).round(2)).reset_index())
savetable(summ, "t01_incidence_by_pair", index=False)
savetable(pooled, "t01_incidence_pooled", index=False)
print(summ.to_string(index=False)); print("\nPOOLED:\n", pooled.to_string(index=False))

saved: t01_incidence_by_pair.csv
saved: t01_incidence_pooled.csv
horizon      pair  linked_n  at_risk  onset  incidence_%
     1y 2019_2020     10687     7001    183         2.61
     1y 2020_2021     10241     6509    156         2.40
     1y 2021_2022      9686     6062    231         3.81
     1y 2022_2023      9348     5758    148         2.57
     1y 2023_2024      9322     5523    183         3.31
     2y 2019_2021      9992     6533    303         4.64
     2y 2020_2022      9427     5996    358         5.97
     2y 2021_2023      9392     5855    360         6.15
     2y 2022_2024      8998     5541    319         5.76

POOLED:
 horizon  at_risk  onset  incidence_pct
     1y    30853    901           2.92
     2y    23925   1340           5.60


In [9]:
# --- Actionable-feature availability audit (for methods/limitations) ---
av=[]
for v in ["BMI","PA_REG","PA_WALK","ALC_FREQ","ALC_AMT","ALC_BINGE"]:
    av.append({"feature":v,"nonnull_%":round(100*panel[v].notna().mean(),1)})
av=pd.DataFrame(av); savetable(av,"t01_actionable_availability", index=False)
print(av.to_string(index=False))

saved: t01_actionable_availability.csv
  feature  nonnull_%
      BMI       92.1
   PA_REG       92.2
  PA_WALK       92.2
 ALC_FREQ       83.4
  ALC_AMT       54.2
ALC_BINGE       54.2
